# OpenAI Structured-Response Evaluation

Evaluate a selected OpenAI model on Calculus I multiple-choice questions. The Responses API is prompted to return JSON, which is parsed and validated locally with Pydantic. Successful responses are checkpointed after each question. Web search is disabled.

## 1. Imports and configuration

In [1]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)

### Run configuration

Change `MODEL_NAME` to select a model compatible with the Responses API. The output directory and filenames follow the model name.

In [2]:
# Support launching Jupyter from either the repository root or notebooks/.
WORKING_DIRECTORY = Path.cwd()
PROJECT_ROOT = (
    WORKING_DIRECTORY.parent
    if WORKING_DIRECTORY.name == "notebooks"
    else WORKING_DIRECTORY
)

MODEL_NAME = "gpt-6-luna"
RUN_NAME = f"{MODEL_NAME}-subsample"

DATA_PATH = PROJECT_ROOT / "data" / "input_data.csv"
MODEL_OUTPUT_DIRECTORY = PROJECT_ROOT / "output" / MODEL_NAME
CHECKPOINT_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-checkpoint.json"
RESULTS_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output.csv"
INCORRECT_RESULTS_PATH = (
    MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output-incorrect.csv"
)

MODEL_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

## 2. Provider clients and response schemas

In [3]:
load_dotenv(PROJECT_ROOT / ".env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), timeout=3600)


class Calculus(BaseModel):
    """Validated response for one calculus question."""

    detailed_solution_explanation: str
    correct_option_choice_letter: str

## 3. Load and inspect the dataset

In [4]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the evaluation dataset."""
    return pd.read_csv(data_path, keep_default_na=False)


def summarize_dataset(dataframe: pd.DataFrame) -> pd.Series:
    """Summarize row counts and duplicate checks used in the original notebook."""
    return pd.Series(
        {
            "rows": len(dataframe),
            "columns": len(dataframe.columns),
            "duplicate_rows": int(dataframe.duplicated().sum()),
            "duplicate_questions": int(dataframe["Question"].duplicated().sum()),
            "duplicate_ids": int(dataframe["ID"].duplicated().sum()),
        }
    )


df = load_dataset(DATA_PATH)
print(f"Loaded {len(df)} questions from {DATA_PATH}")
summarize_dataset(df)

Loaded 331 questions from /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/EDUCAUSE 2026 Conference/calc-llm-bench/data/input_data.csv


rows                   331
columns                  3
duplicate_rows           0
duplicate_questions      0
duplicate_ids            0
dtype: int64

In [5]:
# Inspect any duplicated rows, questions, or IDs found by the summary above.
duplicate_rows = df.loc[df.duplicated(keep=False)].sort_values(df.columns.tolist())
duplicate_questions = df.loc[df["Question"].duplicated(keep=False)].sort_values(
    "Question"
)
duplicate_ids = df.loc[df["ID"].duplicated(keep=False)].sort_values("ID")

df.describe(include="all")

,ID,Question,Correct Answer
count,331,331,331
unique,331,331,7
top,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b
freq,1,1,169


## 4. Primary-model response pipeline

### Few-shot example and system instructions

In [ ]:
EXAMPLE_RESPONSE = """Question:
Complete the square to write the equation in the standard form f(x)=a(x-h)^2+k.
f(x)=2x^2+12x+25
    a) f(x)=2(x+3)^2+16
    b) f(x)=2(x-6)^2+7
    c) f(x)=2(x+3)^2+7
    d) f(x)=2(x-3)^2+16
    e) f(x)=2(x+3)^2-7
    f) None of the above.

Solution:
f(x) = 2(x^2 + 6x) + 25
     = 2((x + 3)^2 - 9) + 25
     = 2(x + 3)^2 + 7.
The calculated solution matches option c.
Final answer: c
"""

SYSTEM_PROMPT = (
    "Act as a mathematician specializing in Calculus. Solve the given "
    "college-level Calculus I multiple-choice problem using the example as a "
    "reference. Provide a clear, accurate, step-by-step explanation of the "
    "solution. Compare the calculated solution with the provided answer "
    "choices and output the matching option letter. If no answer choice "
    'matches the calculated solution, output "NA" as the final answer. '
    'Do not force a match instead of "NA" '
    "when no listed solution matches.\n\nHere is the example:\n"
    + EXAMPLE_RESPONSE
)

RESPONSE_FORMAT_INSTRUCTIONS = (
    "Return only one valid JSON object with the keys "
    "detailed_solution_explanation and correct_option_choice_letter. "
    'Set correct_option_choice_letter to the matching option letter or "NA". '
    "Do not wrap the JSON in Markdown."
)


In [7]:
def load_checkpoint(checkpoint_path: Path) -> dict:
    """Load saved responses, or return an empty checkpoint."""
    if not checkpoint_path.exists():
        return {}
    with checkpoint_path.open("r", encoding="utf-8") as checkpoint_file:
        return json.load(checkpoint_file)


def save_checkpoint(checkpoint_path: Path, data: dict) -> None:
    """Persist responses as formatted JSON."""
    with checkpoint_path.open("w", encoding="utf-8") as checkpoint_file:
        json.dump(data, checkpoint_file, indent=4)

In [8]:
def query_openai(question: str, model: str = MODEL_NAME) -> dict:
    """Request one calculus solution using schema-enforced structured output."""
    if not callable(getattr(getattr(client, "responses", None), "parse", None)):
        raise RuntimeError(
            "This notebook requires an OpenAI Python SDK version that supports "
            "client.responses.parse. Upgrade the openai package and restart the kernel."
        )

    response = client.responses.parse(
        model=model,
        instructions=SYSTEM_PROMPT,
        input="Now solve the following question: " + question,
        text_format=Calculus,
    )
    if response.status != "completed":
        raise RuntimeError(
            f"Response not completed ({response.status}): "
            f"{response.incomplete_details}"
        )
    if response.output_parsed is None:
        raise RuntimeError(
            "No structured solution returned; check the response for a refusal."
        )
    return response.output_parsed.model_dump()

In [9]:
def collect_responses(
    dataframe: pd.DataFrame,
    checkpoint_path: Path,
    model: str = MODEL_NAME,
) -> dict:
    """Query unanswered rows and update the checkpoint incrementally."""
    checkpoint = load_checkpoint(checkpoint_path)

    for _, row in dataframe.iterrows():
        question_id = str(row["ID"])
        if question_id in checkpoint:
            print(f"Skipping ID {question_id}, already processed.")
            continue

        print(f"Processing ID {question_id}...")
        response = query_openai(row["Question"], model=model)
        checkpoint[question_id] = response
        save_checkpoint(checkpoint_path, checkpoint)

    save_checkpoint(checkpoint_path, checkpoint)
    print(f"Checkpoint saved to {checkpoint_path}")
    return checkpoint

### Run the primary-model pipeline

> **Note:** This cell makes billable OpenAI API calls, with web search disabled, only for questions absent from the checkpoint.

In [ ]:
df = df.head(10)  # Limit to first 10 rows for testing

checkpoint = collect_responses(df, CHECKPOINT_PATH)
df.head(10)

Skipping ID q2_sp22_problem10__04, already processed.
Skipping ID q27_sp23_problem04__04, already processed.
Skipping ID q23_sp23_problem09__02, already processed.
Skipping ID q31_sp23_problem03__09, already processed.
Skipping ID q10_sp22_problem07__21, already processed.
Skipping ID q19_fa22_problem03__21, already processed.
Skipping ID q28_sp23_problem03__04, already processed.
Skipping ID q25_sp23_problem08__02, already processed.
Skipping ID q2_sp22_problem06__04, already processed.
Skipping ID q5_sp22_problem07__01, already processed.
Skipping ID q11_sp22_problem07__21, already processed.
Skipping ID q22_sp23_problem09__11, already processed.
Skipping ID q30_sp23_problem03__09, already processed.
Skipping ID q3_sp22_problem10__04, already processed.
Skipping ID q26_sp23_problem04__04, already processed.
Skipping ID q20_fa22_problem06__06, already processed.
Skipping ID q3_sp22_problem06__02, already processed.
Skipping ID q26_sp23_problem12__04, already processed.
Skipping ID q24

,ID,Question,Correct Answer
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c
5,q19_fa22_problem03__21,21) Find the intervals on which \(f(x)=\displa...,c
6,q28_sp23_problem03__04,4) Use differentials to estimate the value of ...,e
7,q25_sp23_problem08__02,2) Differentiate the following: \(f(x)=\displ...,b
8,q2_sp22_problem06__04,4) Find the <i>x</i>- and <i>y</i>-intercepts ...,b
9,q5_sp22_problem07__01,1) Evaluate \(\displaystyle \lim _{x\rightarro...,c


## 5. Prepare answer choices

Read the structured answer fields already saved by the primary model; no extraction-model call is needed.

In [11]:
def add_model_responses(dataframe: pd.DataFrame, checkpoint: dict) -> pd.DataFrame:
    """Add answer choices and explanations from the checkpoint."""
    result = dataframe.copy()
    question_ids = result["ID"].astype(str)
    result["LLM Answer Choice (RAW)"] = question_ids.map(
        lambda question_id: checkpoint.get(question_id, {}).get(
            "correct_option_choice_letter", ""
        )
    )
    result["LLM Answer Explanation"] = question_ids.map(
        lambda question_id: checkpoint.get(question_id, {}).get(
            "detailed_solution_explanation", ""
        )
    )
    return result

### Prepare answer choices from the checkpoint

In [12]:
df = add_model_responses(df, checkpoint)
df.head()

,ID,Question,Correct Answer,LLM Answer Choice (RAW),LLM Answer Explanation
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b,b,Factor out 4 from the quadratic and linear ter...
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b,b,Let x be the side parallel to the dividing fen...
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b,b,Apply the chain rule to f(x)=tan(5^(6x^2)). Th...
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d,d,"By the Fundamental Theorem of Calculus, since ..."
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c,c,The slope of the tangent line at x = -1 is f'(...


## 6. Normalize and evaluate answers

In [13]:
def extract_letter_choice(raw_response: str) -> str:
    """Normalize a letter or NA from plain text or an extraction JSON object."""
    if not isinstance(raw_response, str):
        return ""
    value = raw_response.strip()
    if value.upper() == "NA":
        return "NA"
    if re.fullmatch(r"[a-zA-Z]", value):
        return value.lower()
    try:
        payload = json.loads(value)
    except json.JSONDecodeError:
        return ""
    if not isinstance(payload, dict):
        return ""
    choice = payload.get("correct_option_choice_letter")
    if not isinstance(choice, str):
        return ""
    choice = choice.strip()
    if choice.upper() == "NA":
        return "NA"
    return choice.lower() if re.fullmatch(r"[a-zA-Z]", choice) else ""


def evaluate_answers(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Normalize predictions and calculate weighted classification metrics."""
    result = dataframe.copy()
    result["LLM Answer Choice"] = result["LLM Answer Choice (RAW)"].map(
        extract_letter_choice
    )
    result["Correct"] = (
        result["Correct Answer"] == result["LLM Answer Choice"]
    ).astype(int)

    y_true = result["Correct Answer"]
    y_pred = result["LLM Answer Choice"]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "classification_report": classification_report(
            y_true, y_pred, zero_division=0
        ),
    }
    return result, metrics

In [14]:
df, metrics = evaluate_answers(df)

print(metrics["classification_report"])
print(f"Accuracy:  {metrics['accuracy'] * 100:.2f}%")
print(f"Precision: {metrics['precision'] * 100:.2f}%")
print(f"Recall:    {metrics['recall'] * 100:.2f}%")
print(f"F1 Score:  {metrics['f1'] * 100:.2f}%")
df.head()

              precision    recall  f1-score   support

          NA       0.00      0.00      0.00         0
           a       1.00      1.00      1.00        26
           b       1.00      1.00      1.00       167
           c       1.00      1.00      1.00        41
           d       1.00      0.98      0.99        59
           e       1.00      1.00      1.00        14
           f       1.00      1.00      1.00        17
           g       1.00      1.00      1.00         1

    accuracy                           1.00       325
   macro avg       0.88      0.87      0.87       325
weighted avg       1.00      1.00      1.00       325

Accuracy:  99.69%
Precision: 100.00%
Recall:    99.69%
F1 Score:  99.84%


,ID,Question,Correct Answer,LLM Answer Choice (RAW),LLM Answer Explanation,LLM Answer Choice,Correct
0,q2_sp22_problem10__04,4) Complete the square to write the equation i...,b,b,Factor out 4 from the quadratic and linear ter...,b,1
1,q27_sp23_problem04__04,4) A rectangular playground is to be fenced of...,b,b,Let x be the side parallel to the dividing fen...,b,1
2,q23_sp23_problem09__02,2) Differentiate \(\displaystyle f(x)=\tan \le...,b,b,Apply the chain rule to f(x)=tan(5^(6x^2)). Th...,b,1
3,q31_sp23_problem03__09,9) Given that \(x &gt; 0\) and \( \text{capit...,d,d,"By the Fundamental Theorem of Calculus, since ...",d,1
4,q10_sp22_problem07__21,21) Find the slope of the line that is tangent...,c,c,The slope of the tangent line at x = -1 is f'(...,c,1


## 7. Export results

In [15]:
incorrect_df = df.loc[df["Correct"] == 0].copy()

incorrect_df.to_csv(INCORRECT_RESULTS_PATH, index=False)
df.to_csv(RESULTS_PATH, index=False)

print(f"Saved {len(df)} results to {RESULTS_PATH}")
print(f"Saved {len(incorrect_df)} incorrect results to {INCORRECT_RESULTS_PATH}")

Saved 325 results to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/EDUCAUSE 2026 Conference/calc-llm-bench/output/gpt-6-luna/gpt-6-luna-subsample-test-output.csv
Saved 1 incorrect results to /Users/mansibmursalin/Library/CloudStorage/OneDrive-UniversityOfHouston/CASA LLM Project/EDUCAUSE 2026 Conference/calc-llm-bench/output/gpt-6-luna/gpt-6-luna-subsample-test-output-incorrect.csv
